<a href="https://colab.research.google.com/github/karamdh/arene-des-algos-KARAM-DHIFI/blob/main/jour1_arene.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import load_breast_cancer
import numpy as np

def explorer_dataset():
    """
    Charge le dataset breast_cancer et affiche ses caractéristiques de base.
    - nombre de lignes et colonnes
    - classes possibles et leur répartition
    """
    # Chargement
    data = load_breast_cancer()
    X, y = data.data, data.target

    # Forme
    print(f"Lignes, colonnes : {X.shape}")

    # Répartition des classes
    classes = data.target_names  # ['malignant', 'benign']
    for i, nom in enumerate(classes):
        count = np.sum(y == i)
        pct = count / len(y) * 100
        print(f"Classe {i} ({nom}) : {count} cas ({pct:.1f}%)")

    return X, y

X, y = explorer_dataset()

Lignes, colonnes : (569, 30)
Classe 0 (malignant) : 212 cas (37.3%)
Classe 1 (benign) : 357 cas (62.7%)


In [2]:
# CHECKPOINT 1 — cas normal (déjà fait, on vérifie visuellement)
print("=== CAS NORMAL ===")
X, y = explorer_dataset()

# CHECKPOINT 2 — cas limite : une seule classe
print("\n=== CAS LIMITE (seulement classe 0) ===")
mask = (y == 0)
X_filtre, y_filtre = X[mask], y[mask]
print(f"Lignes, colonnes : {X_filtre.shape}")
classes = load_breast_cancer().target_names
for i, nom in enumerate(classes):
    count = np.sum(y_filtre == i)
    pct = count / len(y_filtre) * 100
    print(f"Classe {i} ({nom}) : {count} cas ({pct:.1f}%)")

# CHECKPOINT 3 — cas adversarial : dataset très déséquilibré simulé
print("\n=== CAS ADVERSARIAL (95% / 5% simulé) ===")
# On garde tous les 0 et seulement 12 cas de 1
idx_0 = np.where(y == 0)[0]        # 212 cas
idx_1 = np.where(y == 1)[0][:12]   # seulement 12 cas bénins
idx_combine = np.concatenate([idx_0, idx_1])
y_dezequilibre = y[idx_combine]

for i, nom in enumerate(classes):
    count = np.sum(y_dezequilibre == i)
    pct = count / len(y_dezequilibre) * 100
    print(f"Classe {i} ({nom}) : {count} cas ({pct:.1f}%)")
print("=> Un modèle naïf qui prédit toujours 'malignant' aurait déjà 94.6% d'accuracy !")

=== CAS NORMAL ===
Lignes, colonnes : (569, 30)
Classe 0 (malignant) : 212 cas (37.3%)
Classe 1 (benign) : 357 cas (62.7%)

=== CAS LIMITE (seulement classe 0) ===
Lignes, colonnes : (212, 30)
Classe 0 (malignant) : 212 cas (100.0%)
Classe 1 (benign) : 0 cas (0.0%)

=== CAS ADVERSARIAL (95% / 5% simulé) ===
Classe 0 (malignant) : 212 cas (94.6%)
Classe 1 (benign) : 12 cas (5.4%)
=> Un modèle naïf qui prédit toujours 'malignant' aurait déjà 94.6% d'accuracy !


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Charger dataset
X, y = explorer_dataset()

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# Normalisation (important pour Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Modèles
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"\n=== {name} ===")

    # Logistic Regression utilise données scalées
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

    print("Accuracy:", acc)
    print(classification_report(y_test, y_pred))

Lignes, colonnes : (569, 30)
Classe 0 (malignant) : 212 cas (37.3%)
Classe 1 (benign) : 357 cas (62.7%)
Train shape: (455, 30)
Test shape: (114, 30)

=== Logistic Regression ===
Accuracy: 0.9824561403508771
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114


=== Random Forest ===
Accuracy: 0.956140350877193
              precision    recall  f1-score   support

           0       0.95      0.93      0.94        42
           1       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



In [4]:
import pandas as pd

# Résultats Phase 2
results = {
    "Logistic Regression": 0.9824561403508771,
    "Random Forest": 0.956140350877193
}

# Conversion en tableau
df_results = pd.DataFrame(list(results.items()), columns=["Modèle", "Accuracy"])
df_results = df_results.sort_values(by="Accuracy", ascending=False)

print("=== CLASSEMENT ARÈNE ===")
print(df_results)

# Champion
champion = df_results.iloc[0]

print("\n CHAMPION DE L'ARÈNE :")
print(f"{champion['Modèle']} avec accuracy = {champion['Accuracy']:.4f}")

=== CLASSEMENT ARÈNE ===
                Modèle  Accuracy
0  Logistic Regression  0.982456
1        Random Forest  0.956140

 CHAMPION DE L'ARÈNE :
Logistic Regression avec accuracy = 0.9825
